**Transform Circuits Data**
1. Read bronze circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId circuit_id, circuitName circuit_name)
4. Rename columns to make them more meaningful (lat latitude, long longitude)
5. Filter out rows where circuit_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver circuits table

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f

folder_name = 'circuits'
file_name   = 'circuits.parquet'
silver_table = f"dev.silver.{folder_name}"

# 1. Read Bronze
raw_df = spark.read.format('parquet')\
              .load(f"{bronze_path}/{folder_name}/{file_name}")

# 2. Cast types + Drop url
raw_df = raw_df.withColumn('lat', f.col('lat').cast('double'))\
               .withColumn('long', f.col('long').cast('double'))\
               .withColumn('ingestion_timestamp', f.col('ingestion_timestamp').cast('timestamp'))\
               .drop('url')                       

# 3. Rename columns
raw_df = raw_df.withColumnRenamed('circuitId',   'circuit_id')\
               .withColumnRenamed('circuitName', 'circuit_name')

# 4. Filter null business key
raw_df = raw_df.filter(f.col('circuit_id').isNotNull())

# 5. Remove duplicates
raw_df = raw_df.dropDuplicates(['circuit_id'])     

# 6. Title Case
raw_df = raw_df.withColumn('circuit_name', f.initcap(f.col('circuit_name')))\
               .withColumn('locality',     f.initcap(f.col('locality')))

# 7. Write to Silver
raw_df.write\
      .format('delta')\
      .mode('overwrite')\
      .option('path',silver_path)\
      .saveAsTable(silver_table)